# Phase 5 — MARLCommunitySolver: Test Notebook

**No Gurobi licence required.**

## Test structure

| Group | Tests | Status |
|-------|-------|--------|
| **A — Interface** | A1–A6 | Run now (no training needed) |
| **B — Behaviour** | B1–B9 | **SKIP** until `train()` is implemented |

Group B tests auto-enable once `marl.is_trained == True`.
To unlock them, implement `MARLCommunitySolver.train()`, `sample()`,
`update_reward()`, `save_model()`, `load_model()` in
`src/hybrid/marl_community_solver.py`.

In [1]:
import sys, os, tempfile
sys.path.insert(0, os.path.abspath('..'))

from src.utils.dataReader import PSTTReader
from src.hybrid.special_constraint_decomposer import SpecialConstraintDecomposer
from src.hybrid.community_solver_base import CommunitySolverBase, MARLCommunitySolverBase
from src.hybrid.marl_community_solver import MARLCommunitySolver
from src.hybrid.random_community_solver import RandomCommunitySolver
import pandas as pd

INSTANCE = '../data/source/instances/agh-fal17.xml'
reader = PSTTReader(INSTANCE)
decomp = SpecialConstraintDecomposer(reader)
result = decomp.classify_classes()
communities = result.communities

marl = MARLCommunitySolver(reader, decomp, seed=42)
print(f'Instance   : {reader.problem_name}')
print(f'Communities: {len(communities)}')
print(f'is_trained : {marl.is_trained}')

Problem Name: agh-fal17, Days: 7, Weeks: 18, Slots/Day: 288
Instance   : agh-fal17
Communities: 31
is_trained : False


---
## A1 — Class hierarchy conformance

In [2]:
assert isinstance(marl, CommunitySolverBase)
assert isinstance(marl, MARLCommunitySolverBase)
print('isinstance checks OK')

base_methods = ['action_space','sample','add_no_good','clear_no_goods','no_good_count','stats','reset_stats']
marl_methods = ['train','update_reward','save_model','load_model']
for m in base_methods + marl_methods:
    assert hasattr(marl, m), f'Missing: {m}'
assert hasattr(marl, 'is_trained')

pd.DataFrame([
    {'Method': m, 'Group': 'CommunitySolverBase'} for m in base_methods
] + [
    {'Method': m, 'Group': 'MARLCommunitySolverBase'} for m in marl_methods
] + [
    {'Method': 'is_trained', 'Group': 'MARLCommunitySolverBase (property)'}
])

isinstance checks OK


,Method,Group
0,action_space,CommunitySolverBase
1,sample,CommunitySolverBase
2,add_no_good,CommunitySolverBase
3,clear_no_goods,CommunitySolverBase
4,no_good_count,CommunitySolverBase
5,stats,CommunitySolverBase
6,reset_stats,CommunitySolverBase
7,train,MARLCommunitySolverBase
8,update_reward,MARLCommunitySolverBase
9,save_model,MARLCommunitySolverBase


---
## A2 — `is_trained` starts `False`

In [3]:
assert marl.is_trained is False
print('A2 OK - is_trained == False before training')

A2 OK - is_trained == False before training


---
## A3 — `action_space`: matches `RandomCommunitySolver`

In [4]:
rand = RandomCommunitySolver(reader, decomp, seed=0)
mismatches = []
for cid in list(result.community)[:30]:
    om = set(marl.action_space(cid))
    or_ = set(rand.action_space(cid))
    if om != or_:
        mismatches.append({'CID': cid, 'MARL': len(om), 'Random': len(or_)})

assert not mismatches, f'Mismatches: {mismatches}'
# Caching check
for cid in list(result.community)[:10]:
    assert marl.action_space(cid) is marl.action_space(cid)
print('A3 OK - action_space identical to RandomCommunitySolver, caching works')

A3 OK - action_space identical to RandomCommunitySolver, caching works


---
## A4 — No-good management

In [5]:
comm0 = communities[0]
dummy = {cid: (0, None) for cid in comm0.class_ids}

assert marl.no_good_count(comm0.id) == 0
marl.add_no_good(comm0.id, dummy)
assert marl.no_good_count(comm0.id) == 1
marl.add_no_good(comm0.id, dummy)   # duplicate
assert marl.no_good_count(comm0.id) == 1
marl.clear_no_goods(comm0.id)
assert marl.no_good_count(comm0.id) == 0
for comm in communities[:3]:
    marl.add_no_good(comm.id, {})
marl.clear_no_goods()
for comm in communities[:3]:
    assert marl.no_good_count(comm.id) == 0

print('A4 OK - add, count, duplicate-guard, clear(id), clear(all) all work')

A4 OK - add, count, duplicate-guard, clear(id), clear(all) all work


---
## A5 — `stats` structure and `reset_stats`

In [6]:
s = marl.stats()
required = ['total_sample_calls','total_attempts','special_constraint_rejections',
            'no_good_rejections','successes','exhausted']
marl_keys = ['train_episodes','mean_reward','update_reward_calls']
for k in required + marl_keys:
    assert k in s, f'Missing: {k}'
print('All required stats keys present')

marl.reset_stats()
s2 = marl.stats()
for k, v in s2.items():
    if k != 'mean_reward':
        assert v == 0
print('A5 OK - reset_stats() zeros all numeric fields')

pd.DataFrame([
    {'Key': k, 'After reset': v, 'Group': 'base' if k in required else 'MARL'}
    for k, v in s2.items()
])

All required stats keys present
A5 OK - reset_stats() zeros all numeric fields


,Key,After reset,Group
0,total_sample_calls,0.0,base
1,total_attempts,0.0,base
2,special_constraint_rejections,0.0,base
3,no_good_rejections,0.0,base
4,successes,0.0,base
5,exhausted,0.0,base
6,train_episodes,0.0,MARL
7,mean_reward,NaN,MARL
8,update_reward_calls,0.0,MARL


---
## A6 — Stub methods raise `NotImplementedError`

In [7]:
comm0 = communities[0]
dummy = {cid: (0, None) for cid in comm0.class_ids}

results_a6 = []
for method_name, call in [
    ('sample',        lambda: marl.sample(comm0, max_attempts=1)),
    ('train',         lambda: marl.train(communities[:1], n_episodes=1)),
    ('save_model',    lambda: marl.save_model('/tmp/test.pt')),
    ('load_model',    lambda: marl.load_model('/tmp/test.pt')),
    ('update_reward', lambda: marl.update_reward(comm0.id, dummy, set(), True)),
]:
    try:
        call()
        results_a6.append({'Method': method_name, 'Raises NotImplementedError': False})
    except NotImplementedError:
        results_a6.append({'Method': method_name, 'Raises NotImplementedError': True})

df_a6 = pd.DataFrame(results_a6)
assert df_a6['Raises NotImplementedError'].all()
# update_reward_calls should increment
assert marl.stats()['update_reward_calls'] == 1
print('A6 OK - all stub methods raise NotImplementedError')
print('     - update_reward_calls counter incremented correctly')
df_a6

A6 OK - all stub methods raise NotImplementedError
     - update_reward_calls counter incremented correctly


,Method,Raises NotImplementedError
0,sample,True
1,train,True
2,save_model,True
3,load_model,True
4,update_reward,True


---
## Group B — Phase 5 Implementation Tests

These cells are gated on `marl.is_trained`.  
They will run automatically once `train()` is implemented and called.

In [8]:
if not marl.is_trained:
    print('Group B SKIPPED -- marl.is_trained is False')
    print('Implement train() in src/hybrid/marl_community_solver.py to enable.')
else:
    print('marl.is_trained == True -- running Group B tests')

Group B SKIPPED -- marl.is_trained is False
Implement train() in src/hybrid/marl_community_solver.py to enable.


In [9]:
# B1 / B8: sample coverage + special constraint satisfaction
if marl.is_trained:
    rows_b = []
    for comm in communities:
        a = marl.sample(comm, max_attempts=500)
        if a is None:
            rows_b.append({'Community': comm.id, 'Result': 'exhausted'})
            continue
        assert set(a.keys()) == comm.class_ids
        ok = decomp.check_special_constraints(comm, a)
        rows_b.append({'Community': comm.id, 'Result': 'satisfied' if ok else 'VIOLATED'})
    df_b = pd.DataFrame(rows_b)
    assert (df_b['Result'] != 'VIOLATED').all()
    print('B1/B8 OK:', df_b['Result'].value_counts().to_dict())
    display(df_b['Result'].value_counts())
else:
    print('B1/B8 SKIP')

B1/B8 SKIP


In [10]:
# B4/B5: train() returns summary with required keys
if marl.is_trained:
    summary = marl.train(communities[:1], n_episodes=1)
    assert 'episodes' in summary and 'mean_reward' in summary
    assert marl.is_trained is True
    print('B4/B5 OK - is_trained=True, train() summary:', summary)
else:
    print('B4/B5 SKIP')

B4/B5 SKIP


In [11]:
# B6: update_reward does not raise after training
if marl.is_trained:
    comm0 = communities[0]
    a0 = marl.sample(comm0, max_attempts=200)
    if a0 is not None:
        marl.update_reward(comm0.id, a0, set(), True)
        marl.update_reward(comm0.id, a0, set(list(comm0.class_ids)[:2]), False)
        print('B6 OK - update_reward() accepted feasible and infeasible signals')
else:
    print('B6 SKIP')

B6 SKIP


In [12]:
# B7: save/load round-trip
if marl.is_trained:
    with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as tf:
        tmp_path = tf.name
    try:
        marl.save_model(tmp_path)
        marl2 = MARLCommunitySolver(reader, decomp, seed=0)
        assert not marl2.is_trained
        marl2.load_model(tmp_path)
        assert marl2.is_trained
        print(f'B7 OK - save/load round-trip: {tmp_path}')
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
else:
    print('B7 SKIP')

B7 SKIP


In [13]:
# B2/B3: no-good rejection + exhaustion
if marl.is_trained:
    comm0 = communities[0]
    s_test = MARLCommunitySolver(reader, decomp, seed=77)
    a = s_test.sample(comm0, max_attempts=500)
    if a is not None:
        s_test.add_no_good(comm0.id, a)
        for _ in range(100):
            a2 = s_test.sample(comm0, max_attempts=500)
            assert a2 != a, 'No-good assignment returned'
        print('B2 OK - no-good rejected in 100 subsequent samples')
    r_none = s_test.sample(comm0, max_attempts=0)
    assert r_none is None
    print('B3 OK - max_attempts=0 returns None')
else:
    print('B2/B3 SKIP')

B2/B3 SKIP


In [14]:
# B9: MARL rejection rate vs Random
if marl.is_trained:
    import matplotlib.pyplot as plt
    rand_b9 = RandomCommunitySolver(reader, decomp, seed=42)
    marl_b9 = MARLCommunitySolver(reader, decomp, seed=42)
    # must train marl_b9 first
    marl_b9.train(communities, n_episodes=50)
    rand_b9.reset_stats(); marl_b9.reset_stats()
    for comm in communities:
        rand_b9.sample(comm, max_attempts=200)
        marl_b9.sample(comm, max_attempts=200)

    rs = rand_b9.stats(); ms = marl_b9.stats()
    rr = rs['special_constraint_rejections'] / max(rs['total_attempts'], 1)
    mr = ms['special_constraint_rejections'] / max(ms['total_attempts'], 1)

    fig, ax = plt.subplots(figsize=(5,3))
    ax.bar(['RandomSolver', 'MARLSolver'], [rr, mr], color=['#2196F3','#FF9800'])
    ax.set_ylabel('Rejection rate')
    ax.set_title('Constraint rejection rate: Random vs MARL')
    plt.tight_layout()
    plt.savefig('phase5_rejection_rate.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'B9: rand={rr:.3f}, marl={mr:.3f}')
    if mr <= rr:
        print('B9 OK - MARL rejection rate <= Random')
    else:
        print(f'B9 TODO - MARL ({mr:.3f}) > Random ({rr:.3f}), needs more training')
else:
    print('B9 SKIP')

B9 SKIP


---
## Summary

In [15]:
summary_rows = [
    ('A1', 'Class hierarchy', 'PASSED'),
    ('A2', 'is_trained initial state', 'PASSED'),
    ('A3', 'action_space matches RandomCommunitySolver', 'PASSED'),
    ('A4', 'No-good management', 'PASSED'),
    ('A5', 'stats structure and reset_stats', 'PASSED'),
    ('A6', 'Stub methods raise NotImplementedError', 'PASSED'),
    ('B1', 'sample() coverage', 'SKIPPED' if not marl.is_trained else 'PASSED'),
    ('B2', 'No-good rejection after training', 'SKIPPED' if not marl.is_trained else 'PASSED'),
    ('B3', 'sample() returns None when exhausted', 'SKIPPED' if not marl.is_trained else 'PASSED'),
    ('B4', 'train() sets is_trained=True', 'SKIPPED' if not marl.is_trained else 'PASSED'),
    ('B5', 'train() summary has required keys', 'SKIPPED' if not marl.is_trained else 'PASSED'),
    ('B6', 'update_reward() post-training', 'SKIPPED' if not marl.is_trained else 'PASSED'),
    ('B7', 'save/load round-trip', 'SKIPPED' if not marl.is_trained else 'PASSED'),
    ('B8', 'sample satisfies special constraints', 'SKIPPED' if not marl.is_trained else 'PASSED'),
    ('B9', 'MARL rejection rate <= Random', 'SKIPPED' if not marl.is_trained else 'CHECK OUTPUT'),
]
df_summary = pd.DataFrame(summary_rows, columns=['Test','Description','Status'])
print(f'A-group: {(df_summary["Status"]=="PASSED").sum()} passed')
print(f'B-group: {(df_summary["Status"]=="SKIPPED").sum()} skipped (need Phase 5 implementation)')
df_summary

A-group: 6 passed
B-group: 9 skipped (need Phase 5 implementation)


,Test,Description,Status
0,A1,Class hierarchy,PASSED
1,A2,is_trained initial state,PASSED
2,A3,action_space matches RandomCommunitySolver,PASSED
3,A4,No-good management,PASSED
4,A5,stats structure and reset_stats,PASSED
5,A6,Stub methods raise NotImplementedError,PASSED
6,B1,sample() coverage,SKIPPED
7,B2,No-good rejection after training,SKIPPED
8,B3,sample() returns None when exhausted,SKIPPED
9,B4,train() sets is_trained=True,SKIPPED
